In [6]:
pip install langchain-groq langgraph

In [7]:
from google.colab import userdata
from langchain_groq import ChatGroq
from langgraph.graph import StateGraph , START , END
from typing import TypedDict

In [8]:
api_key = userdata.get("groq_api_key")

model = ChatGroq(
    model="llama-3.1-8b-instant",
    api_key=api_key
)

In [9]:
class chain_state(TypedDict):
  topic:str
  outline:str
  content:str
  Score:int

In [18]:
# Function define Outline Generator Node
def outline_generate(state:chain_state) -> chain_state:
  topics = state["topic"]
  prompt = f"Generate outline from the following {topics}"

  outline = model.invoke(prompt).content
  state["outline"] = outline
  return state

In [19]:
# Function define Blog Generator Node
def blog_generate(state: chain_state) -> chain_state:
  outline = state["outline"]
  topic = state["topic"]

  prompt = f"Generate blog from the following outline {outline} of this topic{topic}"

  content = model.invoke(prompt).content
  state["content"] = content
  return state

In [12]:
# Function define Blog_Score as Evalutor Generator Node
def blog_score(state:chain_state) -> chain_state:
  outline = state["outline"]
  content = state["content"]

  prompt = f"Evaluate My {content} Score based on the {outline}."

  result = model.invoke(prompt).content
  state["Score"] = result
  return state

In [20]:
# Register as a Graph.
graph = StateGraph(chain_state)

#
graph.add_node("Generate_Outline" , outline_generate)
graph.add_node("Generate_Blog" , blog_generate)
graph.add_node("Grade_myblog_Score" ,  blog_score)

graph.add_edge(START , "Generate_Outline")
graph.add_edge("Generate_Outline" , "Generate_Blog")
graph.add_edge("Generate_Blog" ,"Grade_myblog_Score" )
graph.add_edge("Grade_myblog_Score" , END)

In [23]:
# Ccompile the entire Graph.
workflow = graph.compile()

In [16]:
response = workflow.invoke({"topic" : "Agentic AI in the Pakistan"})
# print(response["outline"])
# print(response["content"])
print(response["Score"])

**Evaluation of the Article: "The Rise of Agentic AI in Pakistan: Opportunities and Challenges"**

**Strengths:**

1. **Comprehensive overview**: The article provides a thorough introduction to Agentic AI, its characteristics, and its applications in various sectors.
2. **Relevant examples**: The author includes specific examples of institutions and companies working on AI projects in Pakistan, making the article more relatable and engaging.
3. **Clear structure**: The article is well-organized and easy to follow, with each section building on the previous one to provide a cohesive narrative.
4. **Balanced perspective**: The author presents both the opportunities and challenges of Agentic AI, providing a balanced view of the topic.

**Weaknesses:**

1. **Lack of depth**: While the article provides a good overview of Agentic AI, it lacks depth in certain areas, such as the technical aspects of AI development and the specifics of regulatory frameworks.
2. **Limited analysis**: The author